# Maker-Checker: Trustworthy QA over Financial Filings

**A question-answering system for SEC filings that knows when *not* to answer.**

---

### The problem

Financial analysts spend enormous time pulling factual answers out of company filings — **capital expenditure, margin trends, whether liabilities are under control**. LLMs can attempt these, but they **hallucinate specific figures** — and in finance, **a confidently wrong number is worse than no answer at all**. A bad figure flows straight into a decision; a deferred question just costs a few minutes.

### The approach

A **maker-checker** pipeline over SEC filings:

- **Maker** — proposes an answer *with a citation*
- **Checker** — independently audits it against the filing
- **Adjudicator** — resolves disagreements, and **abstains when the evidence doesn't clearly support an answer**

The goal: a system that is **trustworthy when it commits**, not one that always answers.

### The benchmark

Evaluated on **FinanceBench** — real questions over public filings (10-Ks, 10-Qs, 8-Ks, earnings releases), each with an analyst-written answer *and* the exact supporting evidence span. It's genuinely hard: the original authors found a strong RAG system got **~81% of questions wrong or refused**. And because it ships **gold evidence spans**, I can measure *grounding and retrieval quality* — not just final-answer accuracy.

### What's inside

- ✅ **A validated judge** — FinanceBench ships no scorer, so I built one and proved it agrees with human grading at **κ = 0.91** before trusting it
- ⚖️ **Maker-checker vs. single-shot baseline** — measuring what the checker actually buys
- 🎯 **Two-axis evaluation** — separating *reasoning quality* (given the evidence) from *retrieval quality* (finding it)

*Scope: FinanceBench open split (150 questions); pipeline evaluated on a 45-question stratified subset, oracle mode, over 3 runs.*

See `DESIGN.md` for the architecture and rationale.

## 1. Setup

API Key and Model Setup

In [9]:
import os, sys

# Keys come from the shell environment -- never hardcode them in the notebook.
#   export ANTHROPIC_API_KEY=...    export MCFB_WORKSPACE_ID=...
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_KEY"  
os.environ.setdefault("MCFB_MAKER_MODEL", "claude-sonnet-4-5")
os.environ.setdefault("MCFB_JUDGE_MODEL", "claude-haiku-4-5")
os.environ.setdefault("MCFB_EMBED_BACKEND", "local")   # local | gemini | openai | voyage | mock
os.environ["MCFB_WORKSPACE_ID"] = "YOUR_WORKSPACE_KEY"   # if your key is org-scoped

sys.path.insert(0, os.path.join(os.getcwd(), "src"))
import pandas as pd
print("env ready; python", sys.version.split()[0])

env ready; python 3.13.9


**Validate the judge against human labels using Cohens Kappa, 
a metric that measures how much better than chance the agreement is, 
kappa > 0.8 is almost perfect**

In [17]:
# ── Section 1: build the validation subset (SAFE: never clobbers your labels) ──
# The generator writes a TEMPLATE. Your labeled file is a separate copy that this
# cell refuses to overwrite once it has labels in it.
import os, shutil, pandas as pd
from data_io import load_questions, stratified_sample
from retrieval import oracle_context
from maker import run_maker
from judge import _looks_numeric_answer
from models import make_llm_call

TEMPLATE = "eval/validation_subset_template.csv"
WORKING  = "eval/validation_subset.csv"

def _has_labels(path):
    if not os.path.exists(path):
        return False
    df = pd.read_csv(path).fillna("")
    return "human_label" in df.columns and (df["human_label"].astype(str).str.strip() != "").any()

# If the working file already has labels, DO NOT regenerate — just load it.
if _has_labels(WORKING):
    val = pd.read_csv(WORKING).fillna("")
    print(f"labeled working file exists ({(val['human_label'].astype(str).str.strip()!='').sum()} labels) "
          f"— skipping generation to protect it.")
else:
    qs = load_questions()
    sample = stratified_sample(qs, n=45, seed=13)
    maker_call = make_llm_call("maker")
    rows = []
    for i, q in enumerate(sample, 1):
        out = run_maker(q.question, oracle_context(q), maker_call)
        rows.append({
            "financebench_id": q.financebench_id, "company": q.company,
            "reasoning_bucket": q.reasoning_bucket,
            "predicted_path": "numeric" if _looks_numeric_answer(q.answer) else "llm",
            "question": q.question, "gold_answer": q.answer,
            "justification": q.justification,
            "model_answer": out.answer if out.can_answer else "[ABSTAIN]",
            "human_label": "",
        })
        print(f"  {i}/{len(rows)}", end="\r")
    val = pd.DataFrame(rows)
    val.to_csv(TEMPLATE, index=False)          # regenerable template
    shutil.copyfile(TEMPLATE, WORKING)          # your copy to label
    print(f"\nwrote template and created {WORKING} to label "
          f"({(val.predicted_path=='numeric').sum()} numeric / {(val.predicted_path=='llm').sum()} llm)")

  45/45
wrote template and created eval/validation_subset.csv to label (16 numeric / 29 llm)


**Imputed values manually in human_label column comparing accuracy of golden_answer to model_answer in response to question column**

In [26]:
import pandas as pd
val = pd.read_csv("eval/validation_subset.csv", skiprows=1).fillna("")
val.columns = val.columns.str.strip()
val["human_label"] = val["human_label"].astype(str).str.strip()
labeled = val[val["human_label"] != ""]
print(f"{len(labeled)}/{len(val)} labeled")

val.to_csv("eval/validation_subset.csv", index=False)   # rewrite WITHOUT the title row

45/45 labeled


In [27]:
from judge import judge_answer
from models import make_llm_call, DEFAULT_JUDGE_MODEL
from metrics import raw_agreement, cohens_kappa, confusion

judge_call = make_llm_call(DEFAULT_JUDGE_MODEL)

human, machine, disagree = [], [], []
for _, r in labeled.iterrows():
    jr = judge_answer(r.question, r.gold_answer, r.get("justification",""),
                      r.model_answer, llm_call=judge_call)
    h = r.human_label.strip().lower()
    human.append(h); machine.append(jr.verdict)
    if (h == "correct") != (jr.verdict == "correct"):
        disagree.append((r.financebench_id, r.predicted_path, h, jr.verdict, jr.reason,
                         r.question[:80], r.gold_answer[:50], r.model_answer[:50]))

hb = ["correct" if x=="correct" else "incorrect" for x in human]
mb = ["correct" if x=="correct" else "incorrect" for x in machine]
kappa = cohens_kappa(hb, mb)
print(f"items:        {len(labeled)}")
print(f"raw agreement:{raw_agreement(hb, mb):.3f}")
print(f"cohen kappa:  {kappa:.3f}  ->  {'PASS, judge trusted' if kappa>=0.8 else 'FAIL, revise rubric/tolerance'}")
print("confusion:    ", confusion(human, machine))

items:        45
raw agreement:0.956
cohen kappa:  0.911  ->  PASS, judge trusted
confusion:     {'tp': 22, 'tn': 21, 'fp': 0, 'fn': 2}


### Judge result

Kappa > 0.911 the score is very acceptable and the judge performs well enough unsupervised.

In [ ]:
dis = pd.DataFrame(disagree, columns=["id","path","human","judge","judge_reason","question","gold","model"])
print(f"{len(dis)} disagreements")
dis

## 2. Retrieval quality (no LLM)

Retrieval finds the relevant passages in the filings before the model reasons over them; I measured that it surfaced the correct evidence 60-70% of the time, which bounds how well the end-to-end system can do.

### grab pdfs with Terminal cd ~/maker_checker_financebench/mcfb
###python - << 'EOF'
import os, sys, urllib.request
sys.path.insert(0, "src")
from data_io import load_questions
os.makedirs("pdfs", exist_ok=True)
docs = sorted({q.doc_name for q in load_questions()})
base = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs/"
for i, d in enumerate(docs, 1):
    dest = f"pdfs/{d}.pdf"
    if os.path.exists(dest):
        continue
    try:
        urllib.request.urlretrieve(base + d + ".pdf", dest)
        print(f"  [{i}/{len(docs)}] {d}")
    except Exception as e:
        print(f"  MISS {d}: {e}")
print("done:", len([f for f in os.listdir('pdfs') if f.endswith('.pdf')]), "pdfs")
EOF###

In [39]:
from retrieval import retrieve
from retrieval_eval import score_retrieval, recall_at_k
import os

MODE = "shared_store"     # or "shared_store"
K = 10

# restrict to questions whose PDF is present locally
def have_pdf(d): return os.path.exists(os.path.join("pdfs", d+".pdf"))
run_qs = [q for q in qs if have_pdf(q.doc_name)]
all_docs = tuple(sorted({q.doc_name for q in run_qs}))
print(f"{len(run_qs)} questions with local PDFs; corpus of {len(all_docs)} filings")

hits, recs = [], []
for i, q in enumerate(run_qs, 1):
    ch = retrieve(q, MODE, k=K,
                  all_docs=all_docs if MODE=="shared_store" else None)
    h = score_retrieval(ch, q.evidence)
    hits.append(h)
    recs.append({"id": q.financebench_id, "bucket": q.reasoning_bucket,
                 "page_hit": h.page_hit, "text_hit": h.text_hit,
                 "containment": round(h.best_containment,3),
                 "top_doc_ok": ch[0].chunk.doc_name==q.doc_name if ch else False})
    print(f"  {i}/{len(run_qs)}", end="\r")

r = recall_at_k(hits)
print(f"\n=== {MODE} k={K} backend={os.environ['MCFB_EMBED_BACKEND']} ===")
for key in ["page_recall","text_recall","either_recall","mean_containment"]:
    print(f"  {key:16} {r[key]:.3f}")
pd.DataFrame(recs).head(12)

150 questions with local PDFs; corpus of 84 filings


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/Users/freddie/maker_checker_financebench/mcfb/src/embeddings.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self._m.get_sentence_embedding_dimension()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/Users/freddie/maker_checker_financebench/mcfb/src/embeddings.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self._m.get_sentence_embedding_dimension()


...

  149/150

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  150/150
=== shared_store k=10 backend=local ===
  page_recall      0.467
  text_recall      0.473
  either_recall    0.587
  mean_containment 0.557


/Users/freddie/maker_checker_financebench/mcfb/src/embeddings.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self._m.get_sentence_embedding_dimension()


,id,bucket,page_hit,text_hit,containment,top_doc_ok
0,financebench_id_03029,extraction,False,False,0.400,False
1,financebench_id_04672,extraction,False,False,0.225,True
2,financebench_id_00499,numerical,False,False,0.350,False
3,financebench_id_01226,numerical,False,False,0.300,False
4,financebench_id_01865,logical,False,False,0.500,False
5,financebench_id_00807,numerical,False,False,0.225,False
6,financebench_id_00941,extraction,False,False,0.136,False
7,financebench_id_01858,logical,False,True,1.000,False
8,financebench_id_02987,numerical,True,True,0.625,True
9,financebench_id_07966,numerical,True,True,0.725,True


## 3. Maker-checker pipeline vs single-shot baseline

Measuring if the maker-checker multi-agent pipeline's impact vs the single-shot baseline model of running the single maker model

In [40]:
# --- Maker-checker pipeline vs single-shot baseline (oracle mode) ---
from data_io import load_questions, stratified_sample
from retrieval import oracle_context
from maker import run_maker
from pipeline import run_pipeline
from judge import judge_answer
from metrics import bootstrap_ci
from models import make_llm_call

qs = load_questions()
sample = stratified_sample(qs, n=45, seed=13)   # same subset as judge validation

# role routing: maker+adjudicator -> maker model; checker+judge -> cheaper judge model
maker_call = make_llm_call("maker")
checker_call = make_llm_call("judge")
adjudicator_call = make_llm_call("maker")
judge_call = make_llm_call("judge")

FLOOR = 0.95
rows = []
for i, q in enumerate(sample, 1):
    ctx = oracle_context(q)

    m = run_maker(q.question, ctx, maker_call)                      # baseline
    base_ans = m.answer if m.can_answer else "[ABSTAIN]"
    base_v = judge_answer(q.question, q.answer, q.justification, base_ans, llm_call=judge_call).verdict

    pr = run_pipeline(q.question, ctx, q.reasoning_bucket,          # full pipeline
                      maker_call, checker_call, adjudicator_call)
    pipe_v = "abstained" if pr.abstained else \
        judge_answer(q.question, q.answer, q.justification, pr.answer, llm_call=judge_call).verdict

    rows.append({"id": q.financebench_id, "bucket": q.reasoning_bucket,
                 "baseline": base_v, "path": pr.path,
                 "abstained": pr.abstained, "pipeline": pipe_v,
                 "checker": pr.checker.verdict if pr.checker else "",
                 "grounded": pr.checker.grounded if pr.checker else ""})
    print(f"  {i}/{len(sample)}", end="\r")

df = pd.DataFrame(rows)
df.to_csv("results/pipeline_eval.csv", index=False)

n = len(df)
base_acc = (df.baseline == "correct").sum()
answered = df[~df.abstained]
ans_correct = (answered.pipeline == "correct").sum()
b = bootstrap_ci(base_acc, n)
a = bootstrap_ci(ans_correct, len(answered)) if len(answered) else (0,0,0)

print(f"\n\n=== maker-checker vs single-shot (oracle, n={n}) ===")
print(f"  single-shot accuracy:       {b[0]:.3f}  [{b[1]:.3f}, {b[2]:.3f}]")
print(f"  pipeline coverage:          {len(answered)/n:.3f}  ({len(answered)}/{n})")
print(f"  pipeline answered-accuracy: {a[0]:.3f}  [{a[1]:.3f}, {a[2]:.3f}]")
print(f"  meets {FLOOR:.0%} floor:            {'yes' if len(answered) and ans_correct/len(answered)>=FLOOR else 'no'}")
print(f"  abstained: maker_declined={ (df.path=='maker_declined').sum() }, "
      f"adjudicator={ (df.path=='adjudicator_abstained').sum() }")

  45/45

=== maker-checker vs single-shot (oracle, n=45) ===
  single-shot accuracy:       0.756  [0.622, 0.867]
  pipeline coverage:          0.622  (28/45)
  pipeline answered-accuracy: 0.821  [0.679, 0.964]
  meets 95% floor:            no
  abstained: maker_declined=4, adjudicator=13


### Debugging note: the parsing bug

During development, the single-shot baseline first measured **~0.47** accuracy, concentrated in low performance and high abstention on *numerical* questions — which looked like a numerical-reasoning weakness in the model.

Investigating the raw maker outputs revealed the real cause: the maker was computing answers correctly but formatting them so a strict JSON parser discarded them (reasoning printed before the JSON object). These correct answers were being silently counted as abstentions.

After fixing the parser (`_extract_last_json` + a "JSON on the final line" prompt in `maker.py`), the baseline rose to **~0.74** and numerical abstentions dropped sharply.

*The lesson: a metric can tell a plausible but false story. Reading raw outputs — not just the aggregate score — is what surfaced that the fault was in parsing, not the model.*

### Error analysis

First run had abstained and matching errors that were potentially through a json formating error.  Fixing it the number of wrong answers that went through went down from 6 to 4.


In [41]:
answered = df[~df.abstained]
wrong_approved = answered[answered.pipeline == "incorrect"]
print(f"{len(wrong_approved)} wrong answers the checker let through")
print(wrong_approved.groupby("bucket").size())
print(df.groupby("bucket")[["abstained"]].mean().round(2))  # abstain rate by type

4 wrong answers the checker let through
bucket
logical      1
numerical    3
dtype: int64
            abstained
bucket               
extraction       0.22
logical          0.31
numerical        0.50


## 4. Reproducibility check

Runs the judge model 5× on an identical prompt at temperature=0 and checks for byte-identical output. It was deterministic — so the run-to-run variance in my results comes from elsewhere (the maker model, tested next), not the judge.

In [42]:
import os, anthropic

model = os.environ["MCFB_JUDGE_MODEL"]   # checker/judge model — the one that flipped
client = anthropic.Anthropic(
    api_key=os.environ["ANTHROPIC_API_KEY"],
    default_headers={"anthropic-workspace-id": os.environ["MCFB_WORKSPACE_ID"]},
)

def once():
    r = client.messages.create(
        model=model,
        max_tokens=300,
        system="You are auditing an answer. Reply with STRICT JSON only: "
               '{"verdict":"agree"|"disagree","reason":"<one sentence>"}',
        messages=[{"role":"user","content":
                   "Question: Is 12.1 the correct inventory turnover if COGS=100 and avg inventory=10? "
                   "Candidate answer: 12.1. Audit it."}],
        extra_body={"temperature": 0},
    )
    # surface whether the model emitted a thinking block (nondeterminism source)
    kinds = [getattr(b, "type", "") for b in r.content]
    text = "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
    return text.strip(), kinds

outs = [once() for _ in range(5)]
for i,(t,k) in enumerate(outs):
    print(i, "| blocks:", k, "|", t[:120])
print("\nall 5 identical:", len(set(t for t,_ in outs)) == 1)

0 | blocks: ['text'] | ```json
{
  "verdict": "agree",
  "reason": "Inventory turnover = COGS / Average Inventory = 100 / 10 = 10, not 12.1, so
1 | blocks: ['text'] | ```json
{
  "verdict": "agree",
  "reason": "Inventory turnover = COGS / Average Inventory = 100 / 10 = 10, not 12.1, so
2 | blocks: ['text'] | ```json
{
  "verdict": "agree",
  "reason": "Inventory turnover = COGS / Average Inventory = 100 / 10 = 10, not 12.1, so
3 | blocks: ['text'] | ```json
{
  "verdict": "agree",
  "reason": "Inventory turnover = COGS / Average Inventory = 100 / 10 = 10, not 12.1, so
4 | blocks: ['text'] | ```json
{
  "verdict": "agree",
  "reason": "Inventory turnover = COGS / Average Inventory = 100 / 10 = 10, not 12.1, so

all 5 identical: True


In [43]:
from maker import run_maker
from retrieval import oracle_context

q = next(x for x in sample if x.financebench_id == "financebench_id_00540")
ctx = oracle_context(q)
for i in range(3):
    m = run_maker(q.question, ctx, maker_call)
    print(i, "| can_answer:", m.can_answer, "| answer:", repr(m.answer[:80]))

0 | can_answer: True | answer: 'Approximately 12.1 times. However, conventional inventory management is not part'
1 | can_answer: True | answer: '12.1 times. While this can be calculated, conventional inventory management metr'
2 | can_answer: True | answer: '12.1 times. However, conventional inventory management is less meaningful for AE'


## 5. Final result (mean over 3 runs)

Because the maker model is not fully deterministic even at temperature=0 (it produces the same figures with varying phrasing and rounding), a single evaluation run is not reproducible. Rather than report an unreproducible point estimate, I ran the full evaluation three times and report each metric as mean ± standard deviation.

In [ ]:
%run scripts/06_run_pipeline_multi.py --n 45 --runs 3 --mode oracle

In [ ]:
run 1/3: baseline=0.76 coverage=0.67 answered_acc=0.80
  run 2/3: baseline=0.73 coverage=0.58 answered_acc=0.89
  run 3/3: baseline=0.73 coverage=0.69 answered_acc=0.78

=== 3-run summary (oracle, n=45) ===
  single-shot accuracy:      0.74 +/- 0.01
  pipeline coverage:         0.64 +/- 0.05
  pipeline answered-accuracy:0.82 +/- 0.05

  items answered every run that flipped correctness: 2
    financebench_id_01079: correct 2/3 runs
    financebench_id_00684: correct 1/3 runs

**The pipeline answers 64% of questions (abstaining on the rest) at ~82% accuracy, versus the single-shot baseline's ~74% on everything — trading coverage for an ~8-point accuracy gain, which is the right tradeoff when a confidently wrong financial answer costs more than a deferred one.**

## Findings & limitations

**The result.** This project built a question-answering *system* for financial filings that can abstain rather than hallucinate a costly wrong answer. It also became a comparison of single-shot vs. multi-agent approaches in a realistic financial setting. The pipeline improved answered-accuracy over the single-shot baseline by **roughly 8 points**, though at n=45 this is *directional rather than conclusive* — and the accuracy gain alone doesn't justify the added cost of the multi-agent pipeline. What justifies it is the **abstention**: the system declines the questions it can't verify and defers them to a human, which is the right tradeoff when a confidently wrong financial figure is more expensive than a deferred one.

**The biggest lesson: a bug disguised as a model weakness.** An output-parsing bug in the maker was silently discarding correct answers, which surfaced in the metrics as poor numerical performance. The danger was that this told a *plausible* false story — numerical reasoning is a known LLM weakness, so it would have been easy to accept and start fine-tuning the wrong thing. The real fault was upstream, in the parsing. **The lesson: a metric can point confidently at the wrong component, so I learned to inspect raw outputs before trusting the score.**

**Where I'd go next.** Given more time, I would validate on a substantially larger set than the n=45 used here (kept small for cost), ideally a held-out split. And since numerical reasoning is the consistent weak spot, the clear next step is giving the maker a **calculator tool**, so arithmetic is executed deterministically rather than generated by the model.